<a href="https://colab.research.google.com/github/GilliardMorandim/mba-tcc-usp-inadimplencia/blob/eda%2Ffeature/analise_sobrevivencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements

In [7]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark
!pip install mlxtend
!pip install lifelines
!pip install imblearn


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 13.3 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=e458c0a704c6d1e5d39e0740dc4abfcacfdc7fe655877cb7fea071cde974c2b4
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma


In [8]:
import matplotlib.pyplot as plt
from google.colab import drive
import os
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np


from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

# Import the required libraries for model training
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, f1_score, precision_score, confusion_matrix, recall_score, roc_auc_score, classification_report,ConfusionMatrixDisplay

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

from pyspark.ml.classification import RandomForestClassifier
#from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from matplotlib.artist import get
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import StratifiedKFold

#Analise de Sobrevivencia
from lifelines import KaplanMeierFitter, WeibullAFTFitter, LogNormalAFTFitter
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Baixar Dados

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when
from pyspark.sql.functions import try_divide

spark = (SparkSession.builder
         .appName("LoadAllCSVs")
         .config("spark.driver.memory", "8g")
         .config("spark.executor.memory", "8g")
         .config("spark.sql.files.maxPartitionBytes", "256m")
         .getOrCreate())

print("Spark iniciado!")

Spark iniciado!


In [4]:
dfs_spark = {}

base_path = "/content/drive/MyDrive/MBA - Ciencia de Dados - USP/dados_tcc/"

files = [f for f in os.listdir(base_path) if f.endswith(".csv")]

for file in files:
    full_path = os.path.join(base_path, file)
    print(f"\n📥 Lendo via PySpark: {file}")

    try:
        df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .csv(full_path))

        key = file.replace(".csv", "")
        dfs_spark[key] = df

        print(f"✔ OK - Linhas (estimado pela Spark): {df.count()} | Colunas: {len(df.columns)}")

    except Exception as e:
        print(f"❌ Erro ao ler {file}: {e}")

print("\nTodos os arquivos foram processados!")

globals().update(dfs_spark)


📥 Lendo via PySpark: pre_aprovado.csv
✔ OK - Linhas (estimado pela Spark): 42171363 | Colunas: 11

📥 Lendo via PySpark: parcelas.csv
✔ OK - Linhas (estimado pela Spark): 1865444 | Colunas: 21

📥 Lendo via PySpark: contratos.csv
✔ OK - Linhas (estimado pela Spark): 382539 | Colunas: 26

📥 Lendo via PySpark: score_credito.csv
✔ OK - Linhas (estimado pela Spark): 266126 | Colunas: 3

📥 Lendo via PySpark: analise_conversao.csv
✔ OK - Linhas (estimado pela Spark): 538023 | Colunas: 12

📥 Lendo via PySpark: analise_conversao_v2.csv
✔ OK - Linhas (estimado pela Spark): 538023 | Colunas: 12

📥 Lendo via PySpark: analise_conversao_v3.csv
✔ OK - Linhas (estimado pela Spark): 533525 | Colunas: 11

📥 Lendo via PySpark: parcelas_main_pd_filtrado.csv
✔ OK - Linhas (estimado pela Spark): 1063816 | Colunas: 17

📥 Lendo via PySpark: Tabela_Hash_Enriquecida.csv
✔ OK - Linhas (estimado pela Spark): 104228 | Colunas: 1

📥 Lendo via PySpark: resultados_modelos_classicos.csv
✔ OK - Linhas (estimado pela Sp

#Referencias

- https://www.kaggle.com/code/henriquebas/people-analytics-turnover-python-lifelines
- https://www.datacamp.com/pt/tutorial/weibull-distribution
- https://statplace.com.br/blog/como-fazer-analise-de-sobrevivencia-na-pratica/

# Distribuição Weibull

In [5]:
parcelas_main_pd_filtrado.show(5, truncate=False)

+--------------------------------------+--------------+---------------+--------------------------------------+--------------------------------------+--------------------------------------+---------------+----------------------------------------------------------------+--------------------------+--------------------------+-------+------------------------------+---------+------------+-----+----------------+--------------------------+
|id_contrato                           |data_pagamento|data_vencimento|id_parcela                            |uuid_cliente                          |id_contrato_original                  |id_contrato_pai|cpf_hash_sha256                                                 |flag_inadimplencia_90_days|flag_inadimplencia_30_days|valor  |valor_financiado_principal_iof|valor_iof|valor_tarifa|month|parcela_norm_0_1|valor_juros_remuneratorios|
+--------------------------------------+--------------+---------------+--------------------------------------+------------------

# Distribuição Log-Normal

# Pipeline

In [13]:
class SurvivalPipeline:
    def __init__(
        self,
        duration_col,   # coluna de tempo (ex: parcelas)
        event_col,      # indicador de evento
        covariates,
        scale_covariates=True
    ):
        self.duration_col = duration_col
        self.event_col = event_col
        self.covariates = covariates
        self.scale_covariates = scale_covariates

        self.scaler = StandardScaler() if scale_covariates else None
        self.models = {}
        self.results = {}

    def _prepare_data(self, df):
        data = df[[self.duration_col, self.event_col] + self.covariates].copy()

        if self.scale_covariates:
            data[self.covariates] = self.scaler.fit_transform(
                data[self.covariates]
            )

        return data

    def fit(self, df):
        data = self._prepare_data(df)

        weibull = WeibullAFTFitter()
        weibull.fit(
            data,
            duration_col=self.duration_col,
            event_col=self.event_col
        )

        lognormal = LogNormalAFTFitter()
        lognormal.fit(
            data,
            duration_col=self.duration_col,
            event_col=self.event_col
        )

        self.models["weibull"] = weibull
        self.models["lognormal"] = lognormal

        self._evaluate_models()

        return self

    def _evaluate_models(self):
        for name, model in self.models.items():
            self.results[name] = {
                "AIC": model.AIC_,
                "LogLikelihood": model.log_likelihood_
            }

    def compare_models(self):
        return pd.DataFrame(self.results).T.sort_values("AIC")

    def summary(self, model_name):
        return self.models[model_name].summary

    def predict_survival(self, model_name, X, times):
        model = self.models[model_name]

        if self.scale_covariates:
            X_scaled = pd.DataFrame(
                self.scaler.transform(X),
                columns=self.covariates
            )
        else:
            X_scaled = X.copy()

        return model.predict_survival_function(X_scaled, times=times)

    def predict_pd(self, model_name, X, times):
        survival = self.predict_survival(model_name, X, times)
        return 1 - survival

    def predict_hazard(self, model_name, X):
        model = self.models[model_name]
        return model.predict_hazard_ratios(X)

    def predict_median(self, model_name, X):
        model = self.models[model_name]
        return model.predict_median(X)

    def predict_mean(self, model_name, X):
        model = self.models[model_name]
        return model.predict_mean(X)

In [ ]:
covariates = [

]

pipeline = SurvivalPipeline(
    duration_col="duration",
    event_col="event",
    covariates=covariates,
    scale_covariates=True
)

pipeline.fit(df)
pipeline.compare_models()